In [ ]:
import os
from pathlib import Path

print("=== /kaggle/input contents ===")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_dir():
        imgs = list(p.glob("*.jpg")) + list(p.glob("*.png"))
        if imgs:
            print(f"  {p.relative_to('/kaggle/input')} -> {len(imgs)} images")


In [ ]:
import shutil
import random
import yaml
import zipfile
from pathlib import Path
from collections import defaultdict, Counter

random.seed(42)

proc_dir = Path('/kaggle/working/data/processed')
if proc_dir.exists():
    shutil.rmtree(proc_dir)

zip_candidates = list(Path('/kaggle/input').rglob('*.zip')) + list(Path('/kaggle/working').rglob('*.zip'))
found_prepackaged = False

for z in zip_candidates:
    try:
        with zipfile.ZipFile(z, 'r') as zf:
            namelist = zf.namelist()
            if any('data.yaml' in n for n in namelist):
                print(f'Found pre-packaged 4-class dataset zip: {z}')
                print('Extracting to /kaggle/working/data/processed...')
                zf.extractall(proc_dir)
                found_prepackaged = True
                break
    except Exception:
        pass

if not found_prepackaged:
    for yml in Path('/kaggle/input').rglob('data.yaml'):
        with open(yml) as yf:
            content = yf.read()
            if 'board' in content and 'chair' in content:
                print(f'Found existing 4-class dataset at {yml.parent}')
                shutil.copytree(yml.parent, proc_dir)
                found_prepackaged = True
                break

if not found_prepackaged:
    print('No pre-packaged dataset found. Merging and remapping raw datasets...')
    for split in ['train', 'val', 'test']:
        (proc_dir / split / 'images').mkdir(parents=True, exist_ok=True)
        (proc_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

    cat_to_id = {
        'board': 0,
        'chair': 1,
        'desk': 2,
        'table': 2,
        'classroom': 2,
        'fan': 3
    }

    input_root = Path('/kaggle/input')
    all_pairs = []

    for img in input_root.rglob('*.*'):
        if img.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            continue
        lbl_cand1 = img.parent.parent / 'labels' / f'{img.stem}.txt'
        lbl_cand2 = img.parent / f'{img.stem}.txt'
        lbl = lbl_cand1 if lbl_cand1.exists() else (lbl_cand2 if lbl_cand2.exists() else None)
        
        if lbl:
            path_str = str(img).lower()
            target_id = None
            target_name = None
            for key, cid in cat_to_id.items():
                if key in path_str:
                    target_id = cid
                    target_name = 'desk' if key in ['table', 'classroom'] else key
                    break
            if target_id is not None:
                all_pairs.append((img, lbl, target_id, target_name))

    print(f'Total paired samples found: {len(all_pairs)}')
    random.shuffle(all_pairs)
    n = len(all_pairs)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.20)

    splits = {
        'train': all_pairs[:n_train],
        'val':   all_pairs[n_train:n_train + n_val],
        'test':  all_pairs[n_train + n_val:]
    }

    cls_counts = Counter()
    for split_name, samples in splits.items():
        for idx, (img_p, lbl_p, target_id, cat) in enumerate(samples):
            new_stem = f'{cat}_{idx:05d}'
            shutil.copy2(img_p, proc_dir / split_name / 'images' / f'{new_stem}{img_p.suffix}')
            
            remapped_lines = []
            with open(lbl_p, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        remapped_lines.append(f'{target_id} ' + ' '.join(parts[1:]))
                        cls_counts[target_id] += 1
            with open(proc_dir / split_name / 'labels' / f'{new_stem}.txt', 'w') as out_f:
                out_f.write('\n'.join(remapped_lines) + '\n')
        print(f'  {split_name}: {len(samples)} samples copied')

data_cfg = {
    'path': '/kaggle/working/data/processed',
    'train': 'train/images',
    'val':   'valid/images' if (proc_dir / 'valid').exists() else 'val/images',
    'test':  'test/images',
    'nc':    4,
    'names': ['board', 'chair', 'desk', 'fan']
}

yaml_path = proc_dir / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f'\ndata.yaml configured: {yaml_path}')
print('Classes (4): [board, chair, desk, fan]')



In [ ]:
from ultralytics import RTDETR

print("=" * 70)
print("RT-DETR-L TRAINING  |  T4 x2  |  30 epochs  |  ~4 hours")
print("=" * 70)

model = RTDETR("rtdetr-l.pt")

results = model.train(
    data="/kaggle/working/data/processed/data.yaml",
    epochs=30,
    imgsz=640,
    batch=32, 
    device=[0, 1],
    patience=15,
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.01,
    warmup_epochs=3,
    project="/kaggle/working/runs/train",
    name="classroom_4class",
    save_period=5,
    seed=42,
    deterministic=True,
    pretrained=True,
    workers=4,
    verbose=True,
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)


In [ ]:
from ultralytics import RTDETR

best_pt = "/kaggle/working/runs/train/classroom_4class/weights/best.pt"

eval_model = RTDETR(best_pt)

metrics = eval_model.val(
    data="/kaggle/working/data/processed/data.yaml",
    split="test",
    conf=0.5,
    iou=0.5,
    save_json=True,
    plots=True
)

print("\n" + "=" * 70)
print("4-CLASS MODEL EVALUATION RESULTS")
print("=" * 70)
print(f"mAP@0.5:       {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")
print("=" * 70)
print("\nPer-class results:")
for i, name in eval_model.names.items():
    r = metrics.box.class_result(i)
    print(f"  {name:10s}  P={r[0]:.3f}  R={r[1]:.3f}  mAP@0.5={r[2]:.3f}")


In [ ]:
import shutil
from pathlib import Path

out_dir = Path("/kaggle/working/output_4class")
out_dir.mkdir(exist_ok=True)

shutil.copy2(
    "/kaggle/working/runs/train/classroom_4class/weights/best.pt",
    out_dir / "best.pt"
)

for png in Path("/kaggle/working/runs/train/classroom_4class").glob("*.png"):
    shutil.copy2(png, out_dir / png.name)

shutil.make_archive("/kaggle/working/classroom_4class_model", "zip", str(out_dir))

print("Output zip ready: /kaggle/working/classroom_4class_model.zip")
print("  -> In Kaggle: click the file in the output panel and download")
